In [1]:
# make_baseline_figures_fast.py
# Baseline figures from ULMM pickles (turbo):
# - H2: ServC (kappa=1) vs B3 e vs median-quantile (tau ∈ {300,600}s)
# - H3 (opzionale): shock con targeting B1/B2 vs random, con multi-source dijkstra e weight callable
#
# Dipendenze: numpy, pandas, networkx, matplotlib

import warnings, pickle, math, heapq
from pathlib import Path
from collections import defaultdict
from typing import Dict, Tuple, List, Iterable, Callable

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# -----------------------------
# Config (tuning per velocità)
# -----------------------------
PICKLES_DIR = Path("ulmm_pickles")
RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR     = Path("fig");     FIG_DIR.mkdir(parents=True, exist_ok=True)

VEHICLE      = "van"
ORIENTATIONS = ["AD", "DA"]
TAUS         = [300, 600]

# Modalità turbo: valori bassi => molto veloce (alza per più precisione)
DO_H2 = True
DO_H3 = True                 # metti False se vuoi solo le coverage

FAST_MODE            = True
QS_POINTS            = 12    # punti sulla curva q∈(0,1]
D_SAMPLE_H2          = 300   # max domande per H2 (None = tutte)
BETW_SAMPLE_SOURCES  = 250   # fonti per B1 (subset)   [↑ = +accurate, +lento]
B2_SOURCES           = 180   # fonti per B2 (subset)
B2_TARGETS_PER_SRC   = 180   # target per ciascuna fonte in B2
RAND_REPS_H3         = 3     # replicati random per shock
BUDGET_PCT           = 1.0   # % archi rimossi
RNG_SEED             = 7

# -----------------------------
# Utilities
# -----------------------------
def slugify(s: str) -> str:
    import re
    return re.sub(r"[^a-z0-9]+","-",str(s).lower()).strip("-")

def find_pickles() -> List[Path]:
    return sorted(PICKLES_DIR.glob("ulmm_*.pkl"))

def load_ulmm(pkl: Path) -> dict:
    with open(pkl, "rb") as f: return pickle.load(f)

def collapsed_graph(ulmm: dict, veh: str) -> nx.DiGraph:
    Gm: nx.MultiDiGraph = ulmm["graph"]
    G = nx.DiGraph()
    wkey = f"c_eff_{veh}"
    add = G.add_edge
    for u, v, k, d in Gm.edges(keys=True, data=True):
        w = d.get(wkey, None)
        if w is None or not np.isfinite(w): continue
        w = float(w)
        if (u, v) not in G or w < G[u][v]["weight"]:
            add(int(u), int(v), weight=w)
    return G

def _maybe_sample_demands(df: pd.DataFrame, max_n: int | None) -> np.ndarray:
    n = len(df)
    if (not FAST_MODE) or (max_n is None) or (n <= max_n):
        return np.arange(n, dtype=int)
    w = np.clip(np.asarray(df["w"], float), 0, None)
    probs = (w / w.sum()) if w.sum() > 0 else None
    rng = np.random.default_rng(RNG_SEED)
    idx = rng.choice(np.arange(n), size=max_n, replace=False, p=probs)
    return np.sort(idx)

# -----------------------------
# φ prox per ServC
# -----------------------------
def _phi_prox_map(ulmm: dict, veh: str, q=0.8) -> dict:
    Gm = ulmm["graph"]; key = f"phi_{veh}"; out = {}
    for a in ulmm["access"]["i_node"].astype(int).tolist():
        vals = []
        try:
            vals += [d.get(key) for *_ , d in Gm.edges(a, keys=True, data=True)]
            vals += [d.get(key) for *_ , d in Gm.in_edges(a, keys=True, data=True)]
        except Exception:
            pass
        vals = [float(v) for v in vals if v is not None and np.isfinite(v)]
        out[a] = float(np.quantile(vals, q)) if vals else 0.0
    return out

# -----------------------------
# Baselines
# -----------------------------
def b1_edge_betweenness(G: nx.DiGraph, k_sources: int) -> Dict[Tuple[int,int], float]:
    nodes = list(G.nodes())
    if not nodes: return {}
    rng = np.random.default_rng(RNG_SEED)
    k = min(int(k_sources), len(nodes))
    src = list(rng.choice(nodes, size=k, replace=False))
    return nx.edge_betweenness_centrality_subset(G, sources=src, targets=nodes,
                                                 weight="weight", normalized=True)

def b2_traversal_counts_sampled(G: nx.DiGraph, s_sources: int, t_per_source: int) -> Dict[Tuple[int,int], float]:
    """
    B2 veloce: per ciascuna fonte campionata, prendi un set di target e conta
    UN percorso di minima per target (ricostruzione dalla pred-list).
    """
    rng = np.random.default_rng(RNG_SEED)
    nodes = list(G.nodes())
    if not nodes: return {}
    S = min(int(s_sources), len(nodes))
    counts = defaultdict(float)
    for s in rng.choice(nodes, size=S, replace=False):
        dist, pred = nx.single_source_dijkstra(G, s, weight="weight")
        reachable = [n for n in dist.keys() if n != s]
        if not reachable: continue
        T = min(int(t_per_source), len(reachable))
        for t in rng.choice(reachable, size=T, replace=False):
            u = t
            while pred[u]:
                v = pred[u][0]
                if G.has_edge(v, u): counts[(v, u)] += 1.0
                u = v
    return counts

def b3_access_closeness(ulmm: dict, veh: str, orientation: str) -> pd.DataFrame:
    demand, access = ulmm["demand"], ulmm["access"]
    d_nodes = demand["i_node"].astype(int).to_numpy()
    a_nodes = access["i_node"].astype(int).to_numpy()
    w_d     = demand["w"].astype(float).to_numpy()

    idxD = _maybe_sample_demands(demand, D_SAMPLE_H2)
    d_nodes = d_nodes[idxD]; w_d = w_d[idxD]

    G = collapsed_graph(ulmm, veh)
    G_use = G if orientation=="AD" else G.reverse(copy=False)

    rows=[]
    for a in a_nodes:
        dist = nx.single_source_dijkstra_path_length(G_use, int(a), weight="weight")
        s=0.0
        for d,wd in zip(d_nodes,w_d):
            t = dist.get(int(d), np.inf)
            if np.isfinite(t) and t>0: s += wd/t
        rows.append((int(a), float(s)))
    return pd.DataFrame(rows, columns=["a_node","b3"])

def servc(ulmm: dict, veh: str, orientation: str, kappa: float=1.0, eps: float=1e-9) -> pd.DataFrame:
    demand, access = ulmm["demand"], ulmm["access"]
    d_nodes = demand["i_node"].astype(int).to_numpy()
    a_nodes = access["i_node"].astype(int).to_numpy()
    w_d     = demand["w"].astype(float).to_numpy()

    idxD = _maybe_sample_demands(demand, D_SAMPLE_H2)
    d_nodes = d_nodes[idxD]; w_d = w_d[idxD]

    G = collapsed_graph(ulmm, veh)
    G_use = G if orientation=="AD" else G.reverse(copy=False)
    phi_prox = _phi_prox_map(ulmm, veh)

    rows=[]
    for a in a_nodes:
        pa = float(phi_prox.get(int(a),0.0))
        dist = nx.single_source_dijkstra_path_length(G_use, int(a), weight="weight")
        s=0.0
        for d,wd in zip(d_nodes, w_d):
            t = dist.get(int(d), np.inf)
            if np.isfinite(t): s += wd/(t + kappa*pa + eps)
        rows.append((int(a), float(s)))
    return pd.DataFrame(rows, columns=["a_node","servc"])

# -----------------------------
# H2 coverage (curvature ridotta)
# -----------------------------
def coverage_curves(ulmm: dict, veh: str, orientation: str,
                    taus: Iterable[int], qs=None) -> pd.DataFrame:
    if qs is None: qs = np.linspace(0.05, 1.0, QS_POINTS)

    df_serv = servc(ulmm, veh, orientation, kappa=1.0)
    df_b3   = b3_access_closeness(ulmm, veh, orientation)
    a_nodes = df_serv["a_node"].astype(int).to_numpy()
    serv    = df_serv["servc"].to_numpy()
    b3      = df_b3["b3"].to_numpy()

    demand = ulmm["demand"]
    d_all = demand["i_node"].astype(int).to_numpy()
    w_all = demand["w"].astype(float).to_numpy()
    idxD  = _maybe_sample_demands(demand, D_SAMPLE_H2)
    d_nodes = d_all[idxD]; w_d = w_all[idxD]

    G = collapsed_graph(ulmm, veh)
    G_use = G if orientation=="AD" else G.reverse(copy=False)

    # Precompute dist(a -> d) per access (30-300 Dijkstra: ok)
    dist_maps=[]
    for a in a_nodes:
        dist = nx.single_source_dijkstra_path_length(G_use, int(a), weight="weight")
        dist_maps.append({int(d): dist.get(int(d), np.inf) for d in d_nodes})

    order_median = np.argsort(np.abs(serv - np.median(serv)))

    def _cov(idxA, tau):
        tau_min = np.full(len(d_nodes), np.inf, float)
        for j in idxA:
            mm = dist_maps[j]
            tau_j = np.array([mm.get(int(d), np.inf) for d in d_nodes], float)
            tau_min = np.minimum(tau_min, tau_j)
        mask = np.isfinite(tau_min) & (tau_min <= tau)
        return float(w_d[mask].sum())/float(w_d.sum()) if w_d.sum()>0 else np.nan

    rows=[]
    for tau in taus:
        for q in qs:
            k = max(1, int(np.ceil(q*len(a_nodes))))
            idx_serv   = np.argsort(serv)[::-1][:k]
            idx_b3     = np.argsort(b3)[::-1][:k]
            idx_median = order_median[:k]
            rows.append(dict(city=ulmm["city"], vehicle=veh, orientation=orientation, tau=int(tau), q=float(q),
                             cov_servc=_cov(idx_serv, tau),
                             cov_b3=_cov(idx_b3, tau),
                             cov_median=_cov(idx_median, tau)))
    return pd.DataFrame(rows)

# -----------------------------
# H3 shock (baseline-only) con weight callable (no copie grafo)
# -----------------------------
def _nearest_access_times(G: nx.DiGraph, access_nodes: np.ndarray, ignore_edges: set | None) -> Dict[int, float]:
    # weight function: rende "infinite" gli archi rimossi
    def w(u, v, d):
        if ignore_edges and (u, v) in ignore_edges: return math.inf
        return d.get("weight", 1.0)
    # multi-source dijkstra path length (no super-source)
    return nx.multi_source_dijkstra_path_length(G, sources=[int(a) for a in access_nodes], weight=w)

def _impact_stats(G, a_nodes, d_nodes, tau0, ignore):
    # baseline già in tau0; recompute after removal
    import numpy as np, math
    tau1 = _nearest_access_times(G, a_nodes, ignore=ignore)
    rel_all, rel_aff = [], []
    n0 = 0
    for d in d_nodes:
        t0 = tau0.get(int(d), math.inf)
        if not np.isfinite(t0): 
            continue
        n0 += 1
        t1 = tau1.get(int(d), math.inf)
        if np.isfinite(t1):
            r = (t1 - t0)/max(t0, 1e-9)
            rel_all.append(r)
            if r > 0: rel_aff.append(r)
        else:
            # isolazione → impatto “molto grande”: cap a 5x per robustezza
            rel_all.append(5.0)
            rel_aff.append(5.0)
    if n0 == 0:
        return dict(med=np.nan, share_aff=np.nan, med_aff=np.nan, q75_aff=np.nan)
    share_aff = len(rel_aff)/n0
    med = float(np.median(rel_all))*100.0 if rel_all else np.nan
    med_aff = float(np.median(rel_aff))*100.0 if rel_aff else np.nan
    q75_aff = float(np.percentile(rel_aff, 75))*100.0 if rel_aff else np.nan
    return dict(med=med, share_aff=share_aff, med_aff=med_aff, q75_aff=q75_aff)

def shock_sensitivity_baseline(ulmm: dict, veh: str,
                               targeting: str, budget_pct: float, n_rand: int) -> pd.DataFrame:
    G = collapsed_graph(ulmm, veh)
    E_all = list(G.edges())
    if not E_all: return pd.DataFrame(columns=["city","vehicle","orientation","target","T_med","R_med"])

    rng = np.random.default_rng(RNG_SEED)

    # baseline scores
    if targeting.upper()=="B1":
        scores = b1_edge_betweenness(G, BETW_SAMPLE_SOURCES)
    else:
        scores = b2_traversal_counts_sampled(G, B2_SOURCES, B2_TARGETS_PER_SRC)
    scores = {e: float(v) for e, v in scores.items() if G.has_edge(*e)}
    n_remove = max(1, int(round(len(scores)*(budget_pct/100.0))))
    E_sorted = [e for e,_ in sorted(scores.items(), key=lambda kv: kv[1], reverse=True)]
    top_edges = set(E_sorted[:n_remove])

    # random replicates (set di archi)
    rand_sets = [ set(map(tuple, rng.choice(E_all, size=n_remove, replace=False))) for _ in range(n_rand) ]

    demand, access = ulmm["demand"], ulmm["access"]
    d_all = demand["i_node"].astype(int).to_numpy()
    a_nodes = access["i_node"].astype(int).to_numpy()
    idxD = _maybe_sample_demands(demand, D_SAMPLE_H2)
    d_nodes = d_all[idxD]

    rows=[]
    for ori in ORIENTATIONS:
        G_use = G if ori=="AD" else G.reverse(copy=False)
        # τ0 una sola volta per orientazione
        tau0_map = _nearest_access_times(G_use, a_nodes, ignore_edges=None)

        T_med = _conditional_median_delta_cached(G_use, a_nodes, d_nodes, tau0_map, ignore_edges=top_edges)
        R_meds = [ _conditional_median_delta_cached(G_use, a_nodes, d_nodes, tau0_map, ignore_edges=rs)
                   for rs in rand_sets ]
        rows.append(dict(city=ulmm["city"], vehicle=veh, orientation=ori, target=targeting.upper(),
                         T_med=T_med, R_med=float(np.nanmedian(R_meds)) if R_meds else np.nan))
    return pd.DataFrame(rows)

# -----------------------------
# Plotting
# -----------------------------
def plot_h2_grid(df_cov: pd.DataFrame, base_key: str, tau: int, out_path: Path):
    sub = df_cov[df_cov["tau"]==tau].copy()
    if sub.empty: return
    cities = sorted(sub["city"].unique())
    n = len(cities); ncols = 2; nrows = int(np.ceil(n/ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(10, 2.6*nrows))
    axes = np.array(axes).reshape(nrows, ncols)

    for i, city in enumerate(cities):
        ax = axes[i//ncols, i%ncols]
        for ori in ORIENTATIONS:
            ss = sub[(sub.city==city) & (sub.orientation==ori)].sort_values("q")
            if ss.empty: continue
            ax.plot(ss["q"], ss["cov_servc"], label=f"{ori} ServC")
            ax.plot(ss["q"], ss[base_key], linestyle="--", label=f"{ori} {base_key.split('_')[1]}")
        ax.set_title(f"{city} — τ={int(tau/60)} min")
        ax.set_xlabel("Access quantile q"); ax.set_ylabel("Weighted coverage")
        ax.set_ylim(0,1.02); ax.grid(True, linewidth=0.5, alpha=0.25)
    for j in range(i+1, nrows*ncols):
        axes[j//ncols, j%ncols].axis("off")
    handles, labels = axes[0,0].get_legend_handles_labels()
    if handles: fig.legend(handles, labels, loc="lower center", ncol=4)
    fig.suptitle(f"H2: ServC vs {base_key.split('_')[1]} (baseline) — τ={int(tau/60)} min", y=0.995)
    plt.tight_layout(rect=[0,0.04,1,0.97])
    fig.savefig(out_path, dpi=180); plt.close(fig)
    print(f"[H2] Saved {out_path}")

def plot_h3_pooled(df_shock: pd.DataFrame, tag: str, out_path: Path):
    if df_shock.empty: return
    agg = df_shock.groupby("orientation")[["T_med","R_med"]].median().reindex(ORIENTATIONS)
    fig, ax = plt.subplots(figsize=(6,4))
    x = np.arange(len(agg))
    ax.bar(x-0.15, agg["T_med"], width=0.3, label="Targeted (T)")
    ax.bar(x+0.15, agg["R_med"], width=0.3, label="Random (R)")
    ax.set_xticks(x); ax.set_xticklabels(agg.index)
    ax.set_ylabel("Median Δτ (%)")
    ax.set_title(f"H3 (baseline): {tag}-targeted vs random — pooled")
    ax.legend(); plt.tight_layout()
    fig.savefig(out_path, dpi=180); plt.close(fig)
    print(f"[H3] Saved {out_path}")

# -----------------------------
# Main
# -----------------------------
def main():
    pickles = find_pickles()
    if not pickles:
        print("Nessun pickle in", PICKLES_DIR); return

    cov_all = []
    shock_b1_all, shock_b2_all = [], []

    for p in pickles:
        ulmm = load_ulmm(p)
        city = ulmm["city"]
        if ulmm["access"].empty or ulmm["demand"].empty:
            print(f"[skip] {city}: access o demand vuoti."); continue
        print(f"[city] {city} — vehicle={VEHICLE}")

        if DO_H2:
            for ori in ORIENTATIONS:
                cov_all.append(coverage_curves(ulmm, VEHICLE, ori, taus=TAUS))

        if DO_H3:
            try:
                shock_b1_all.append(shock_sensitivity_baseline(ulmm, VEHICLE, targeting="B1",
                                                               budget_pct=BUDGET_PCT, n_rand=RAND_REPS_H3))
            except Exception as e:
                print(f"[warn] B1 shock {city}: {e}")
            try:
                shock_b2_all.append(shock_sensitivity_baseline(ulmm, VEHICLE, targeting="B2",
                                                               budget_pct=BUDGET_PCT, n_rand=RAND_REPS_H3))
            except Exception as e:
                print(f"[warn] B2 shock {city}: {e}")

    # ---- save & plot
    if cov_all:
        df_cov = pd.concat(cov_all, ignore_index=True)
        df_cov.to_csv(RESULTS_DIR / f"h2_baseline_covpoints_fast_{slugify(VEHICLE)}.csv", index=False)
        for tau in TAUS:
            plot_h2_grid(df_cov, base_key="cov_b3",
                         tau=tau, out_path=FIG_DIR / f"fig-servc-vsB3_tau{tau}_{slugify(VEHICLE)}.png")
            plot_h2_grid(df_cov, base_key="cov_median",
                         tau=tau, out_path=FIG_DIR / f"fig-servc-vsMedian_tau{tau}_{slugify(VEHICLE)}.png")
    else:
        print("[H2] Nessun dato di coverage.")

    if DO_H3 and shock_b1_all:
        df_b1 = pd.concat(shock_b1_all, ignore_index=True)
        df_b1.to_csv(RESULTS_DIR / f"h3_baseline_shock_B1_fast_{slugify(VEHICLE)}.csv", index=False)
        plot_h3_pooled(df_b1, tag="B1", out_path=FIG_DIR / f"fig-shock-baseline-B1_{slugify(VEHICLE)}.png")
    if DO_H3 and shock_b2_all:
        df_b2 = pd.concat(shock_b2_all, ignore_index=True)
        df_b2.to_csv(RESULTS_DIR / f"h3_baseline_shock_B2_fast_{slugify(VEHICLE)}.csv", index=False)
        plot_h3_pooled(df_b2, tag="B2", out_path=FIG_DIR / f"fig-shock-baseline-B2_{slugify(VEHICLE)}.png")

#with warnings.catch_warnings():
#    warnings.simplefilter("ignore")
#    main()

In [15]:
# ulmm_fastcore.py
import warnings, pickle, math
from pathlib import Path
from collections import defaultdict
from typing import Dict, Tuple, List, Iterable

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# -----------------------------
# Config
# -----------------------------
PICKLES_DIR = Path("ulmm_pickles")
RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR     = Path("fig");     FIG_DIR.mkdir(parents=True, exist_ok=True)

VEHICLE      = "van"
ORIENTATIONS = ["AD", "DA"]
TAUS         = [300, 600]            # 5, 10 minuti

# Velocità/accuratezza
D_SAMPLE_H2  = 400                   # domande max per ServC/coverage
QS_POINTS    = 12
BUDGET_PCT   = 1.5                   # H3: % archi rimossi (del set con score)
RAND_REPS    = 3
RNG_SEED     = 7

# -----------------------------
# Helpers
# -----------------------------
def slugify(s: str) -> str:
    import re
    return re.sub(r"[^a-z0-9]+","-",str(s).lower()).strip("-")

def find_pickles() -> List[Path]:
    return sorted(PICKLES_DIR.glob("ulmm_*.pkl"))

def load_ulmm(pkl: Path) -> dict:
    with open(pkl, "rb") as f: 
        return pickle.load(f)

def collapsed_graph(ulmm: dict, veh: str) -> nx.DiGraph:
    """Collassa MultiDiGraph a DiGraph mantenendo l'arco (u,v) minimo per c_eff_<veh>."""
    Gm: nx.MultiDiGraph = ulmm["graph"]
    G = nx.DiGraph()
    wkey = f"c_eff_{veh}"
    add = G.add_edge
    for u, v, k, d in Gm.edges(keys=True, data=True):
        w = d.get(wkey, None)
        if w is None or not np.isfinite(w): 
            continue
        w = float(w)
        if (u,v) not in G or w < G[u][v]["weight"]:
            add(int(u), int(v), weight=w)
    return G

def _maybe_sample_demands(df: pd.DataFrame, max_n: int | None) -> np.ndarray:
    n = len(df)
    if max_n is None or n <= max_n: 
        return np.arange(n, dtype=int)
    w = np.clip(np.asarray(df["w"], float), 0, None)
    probs = (w / w.sum()) if w.sum()>0 else None
    rng = np.random.default_rng(RNG_SEED)
    idx = rng.choice(np.arange(n), size=max_n, replace=False, p=probs)
    return np.sort(idx)

# -----------------------------
# NAE: Nearest-Access Exposure (una Dijkstra per orientazione)
# -----------------------------
def nae_edge_scores(ulmm: dict, veh: str, orientation: str) -> pd.DataFrame:
    """
    Foresta dei cammini minimi verso l'accesso più vicino (orientata).
    Conta gli archi usati dai cammini domanda->accesso più vicino, pesando per w_d.
    """
    G = collapsed_graph(ulmm, veh)
    if orientation == "DA":
        G = G.reverse(copy=False)

    access = ulmm["access"]; demand = ulmm["demand"]
    A = access["i_node"].astype(int).tolist()
    D = demand["i_node"].astype(int).tolist()
    W = demand["w"].astype(float).to_numpy()

    if not A or not D or len(G) == 0:
        return pd.DataFrame(columns=["u","v","nae","orientation","city","vehicle","nae_norm"])

    # super-source S -> A con peso 0
    S = max(G.nodes) + 1
    H = G.copy()
    for a in A: 
        H.add_edge(S, int(a), weight=0.0)

    # predecessor & distance (una run)
    pred, dist = nx.dijkstra_predecessor_and_distance(H, S, weight="weight")
    pred = {int(k): [int(x) for x in v] for k, v in pred.items()}

    # conta archi lungo il cammino verso l'accesso più vicino
    counts = defaultdict(float)
    for d_idx, d_node in enumerate(D):
        u = int(d_node)
        if u not in pred: 
            continue
        w = float(W[d_idx])
        while pred.get(u):
            v = pred[u][0]
            if v == S: 
                break
            if H.has_edge(v, u):
                counts[(v, u)] += w
            u = v

    if not counts:
        return pd.DataFrame(columns=["u","v","nae","orientation","city","vehicle","nae_norm"])

    rows = [(int(u), int(v), float(s)) for (u,v), s in counts.items()]
    df = pd.DataFrame(rows, columns=["u","v","nae"])
    df["orientation"] = orientation
    df["city"] = ulmm["city"]
    df["vehicle"] = veh
    df["nae_norm"] = df["nae"] / df["nae"].sum()
    return df

# -----------------------------
# ServC & coverage (H2)
# -----------------------------
def _phi_prox_map(ulmm: dict, veh: str) -> Dict[int, float]:
    Gm = ulmm["graph"]; key = f"phi_{veh}"
    out={}
    for a in ulmm["access"]["i_node"].astype(int).tolist():
        ph=[]
        try:
            for _,_,_,d in Gm.edges(int(a), keys=True, data=True):
                v=d.get(key); 
                if v is not None and np.isfinite(v): ph.append(float(v))
            for _,_,_,d in Gm.in_edges(int(a), keys=True, data=True):
                v=d.get(key); 
                if v is not None and np.isfinite(v): ph.append(float(v))
        except Exception: 
            pass
        out[int(a)] = float(np.mean(ph)) if ph else 0.0
    return out

def _tau50_nearest_access(G: nx.DiGraph, a_nodes: np.ndarray, d_nodes: np.ndarray) -> float:
    """
    Tempo mediano verso l'accesso più vicino (s) sui soli demand passati.
    Una run di multi-source Dijkstra.
    """
    if len(G) == 0 or len(a_nodes) == 0 or len(d_nodes) == 0:
        return 60.0
    tau = nx.multi_source_dijkstra_path_length(G, sources=[int(a) for a in a_nodes], weight="weight")
    vals = [float(tau.get(int(d), np.inf)) for d in d_nodes]
    vals = [v for v in vals if np.isfinite(v)]
    return float(np.median(vals)) if vals else 60.0

def servc(ulmm: dict,
          veh: str,
          orientation: str,
          kappa_mode: str = "auto",   # "auto" o "fixed"
          kappa_s: float = 60.0,      # additivo in secondi per φ̄=1 (se fixed)
          kappa_alpha: float = 0.15,  # se auto: kappa_s = alpha * tau50
          zeta: float = 0.75,         # moltiplicativo: (1+zeta*φ̄)*τ
          gamma_alpha: float = 0.0,   # alpha(a)=exp(-gamma*φ̄)
          eps: float = 1e-9) -> pd.DataFrame:
    """
    Oriented ServC con penalità di prossimità al bordo:
      tau_pen = (1 + zeta*phi_prox(a)) * tau + kappa_s*phi_prox(a)
      alpha(a) = exp(-gamma_alpha*phi_prox(a))
    - kappa_mode="auto": kappa_s = kappa_alpha * tau50 (tempo mediano verso accesso più vicino)
    Ritorna DataFrame [a_node, servc].
    """
    demand = ulmm["demand"]; access = ulmm["access"]
    d_nodes_all = demand["i_node"].astype(int).to_numpy()
    a_nodes = access["i_node"].astype(int).to_numpy()
    w_all     = demand["w"].astype(float).to_numpy()

    # sotto-campionamento per velocità
    idxD = _maybe_sample_demands(demand, D_SAMPLE_H2)
    d_nodes = d_nodes_all[idxD]; w_d = w_all[idxD]

    G = collapsed_graph(ulmm, veh)
    G_use = G if orientation == "AD" else G.reverse(copy=False)

    # kappa auto calibrato
    if kappa_mode == "auto":
        tau50 = _tau50_nearest_access(G_use, a_nodes, d_nodes)
        kappa_s_eff = float(kappa_alpha * tau50)
    else:
        kappa_s_eff = float(kappa_s)

    # φ̄_prox(a)
    phi_prox = _phi_prox_map(ulmm, veh)

    rows = []
    for a in a_nodes:
        pa = float(phi_prox.get(int(a), 0.0))
        alpha_a = float(np.exp(-gamma_alpha * pa)) if gamma_alpha > 0 else 1.0

        dist = nx.single_source_dijkstra_path_length(G_use, int(a), weight="weight")

        s = 0.0
        mult = 1.0 + zeta * pa
        addt = kappa_s_eff * pa
        for d, wd in zip(d_nodes, w_d):
            t = dist.get(int(d), np.inf)
            if np.isfinite(t):
                t_pen = mult * t + addt
                s += wd / max(t_pen, eps)
        rows.append((int(a), float(alpha_a * s)))

    return pd.DataFrame(rows, columns=["a_node","servc"])

def b3_access_closeness(ulmm: dict, veh: str, orientation: str) -> pd.DataFrame:
    """
    Baseline B3: somma delle inverse dei tempi (nessuna penalità di curb-prox).
    Implementato come ServC con zeta=kappa=gamma=0.
    """
    df = servc(ulmm, veh, orientation, kappa_mode="fixed", kappa_s=0.0, zeta=0.0, gamma_alpha=0.0)
    df.rename(columns={"servc":"b3"}, inplace=True)
    return df

def coverage_curves(ulmm, veh, orientation, taus, qs=None, d_max=D_SAMPLE_H2,
                    kappa_mode="auto", kappa_s=60.0, kappa_alpha=0.15,
                    zeta=0.75, gamma_alpha=0.0):
    """
    Copertura pesata al crescere del quantile q per:
     - top-q ServC (penalità curb-prox)
     - top-q B3 (closeness)
     - set "median-quantile" (controllo)
    """
    if qs is None:
        qs = np.linspace(0.05, 1.0, QS_POINTS)

    # ServC consigliato per i grafici del paper
    df_serv = servc(
        ulmm, veh, orientation,
        kappa_mode="fixed",   # <<— importante
        kappa_s=120.0,
        zeta=1.25,
        gamma_alpha=0.5
    )
    df_b3   = b3_access_closeness(ulmm, veh, orientation)

    a_nodes = df_serv["a_node"].astype(int).to_numpy()
    serv    = df_serv["servc"].to_numpy()
    b3      = df_b3["b3"].to_numpy()

    demand = ulmm["demand"]
    d_all = demand["i_node"].astype(int).to_numpy()
    w_all = demand["w"].astype(float).to_numpy()
    idxD  = _maybe_sample_demands(demand, d_max)
    d_nodes = d_all[idxD]; w_d = w_all[idxD]

    G = collapsed_graph(ulmm, veh)
    if orientation == "DA":
        G = G.reverse(copy=False)

    # distanze per ogni accesso (numero accessi ~ 30–300)
    dist_maps=[]
    for a in a_nodes:
        dist = nx.single_source_dijkstra_path_length(G, int(a), weight="weight")
        dist_maps.append({int(d): dist.get(int(d), np.inf) for d in d_nodes})

    order_median = np.argsort(np.abs(serv - np.median(serv)))

    def _cov(idxA, tau):
        tau_min = np.full(len(d_nodes), np.inf, float)
        for j in idxA:
            mm = dist_maps[j]
            tau_j = np.array([mm.get(int(d), np.inf) for d in d_nodes], float)
            tau_min = np.minimum(tau_min, tau_j)
        mask = np.isfinite(tau_min) & (tau_min <= tau)
        return float(w_d[mask].sum())/float(w_d.sum()) if w_d.sum()>0 else np.nan

    rows=[]
    for tau in taus:
        for q in qs:
            k = max(1, int(np.ceil(q*len(a_nodes))))
            idx_serv   = np.argsort(serv)[::-1][:k]
            idx_b3     = np.argsort(b3)[::-1][:k]
            idx_median = order_median[:k]
            rows.append(dict(city=ulmm["city"], vehicle=veh, orientation=orientation, tau=int(tau), q=float(q),
                             cov_servc=_cov(idx_serv, tau),
                             cov_b3=_cov(idx_b3, tau),
                             cov_median=_cov(idx_median, tau)))
    return pd.DataFrame(rows)

# -----------------------------
# H3 shock con NAE (weight callable; nessuna copia grafo)
# -----------------------------
def _nearest_access_times(G: nx.DiGraph, access_nodes: np.ndarray, ignore: set | None) -> Dict[int,float]:
    def w(u,v,d):
        if ignore and (u,v) in ignore: 
            return math.inf
        return d.get("weight", 1.0)
    return nx.multi_source_dijkstra_path_length(G, sources=[int(a) for a in access_nodes], weight=w)

def _impact_stats(G, a_nodes, d_nodes, tau0, ignore, eps_rel=0.005):  # 0.5% soglia
    tau1 = _nearest_access_times(G, a_nodes, ignore=ignore)
    diffs = []
    for d in d_nodes:
        t0 = tau0.get(int(d), math.inf); t1 = tau1.get(int(d), math.inf)
        if np.isfinite(t0) and np.isfinite(t1):
            diffs.append((t1 - t0) / max(t0, 1e-9))
    if not diffs:
        return dict(med=np.nan, share_aff=np.nan, med_aff=np.nan, q75_aff=np.nan)
    diffs = np.array(diffs)
    aff = diffs >= eps_rel
    share_aff = 100.0 * aff.mean()
    med = 100.0 * np.median(diffs)
    med_aff = 100.0 * np.median(diffs[aff]) if aff.any() else np.nan
    q75_aff = 100.0 * np.percentile(diffs[aff], 75) if aff.any() else np.nan
    return dict(med=med, share_aff=share_aff, med_aff=med_aff, q75_aff=q75_aff)

def _conditional_median(G: nx.DiGraph, a_nodes: np.ndarray, d_nodes: np.ndarray,
                        tau0: Dict[int,float], ignore: set | None) -> float:
    tau1 = _nearest_access_times(G, a_nodes, ignore=ignore)
    rel=[]
    for d in d_nodes:
        t0 = tau0.get(int(d), math.inf); t1 = tau1.get(int(d), math.inf)
        if np.isfinite(t0) and np.isfinite(t1):
            rel.append((t1 - t0)/max(t0, 1e-9))
    return np.median(rel)*100.0 if rel else np.nan

def shock_with_nae(ulmm: dict, veh: str, budget_pct=BUDGET_PCT, reps=RAND_REPS) -> pd.DataFrame:
    G = collapsed_graph(ulmm, veh)
    demand, access = ulmm["demand"], ulmm["access"]
    d_nodes = demand["i_node"].astype(int).to_numpy()
    a_nodes = access["i_node"].astype(int).to_numpy()
    rng = np.random.default_rng(RNG_SEED)
    rows = []

    nae = {"AD": nae_edge_scores(ulmm, veh, "AD"),
           "DA": nae_edge_scores(ulmm, veh, "DA")}

    for ori in ["AD","DA"]:
        nae_df = nae[ori]
        if nae_df.empty:
            rows.append(dict(city=ulmm["city"], vehicle=veh, orientation=ori,
                             T_med=np.nan, R_med=np.nan,
                             T_share=np.nan, R_share=np.nan,
                             T_med_aff=np.nan, R_med_aff=np.nan,
                             T_q75_aff=np.nan, R_q75_aff=np.nan))
            continue

        G_use = G if ori=="AD" else G.reverse(copy=False)
        cand = [(int(r.u), int(r.v), float(r.nae)) for r in nae_df.itertuples(index=False)
                if r.nae>0 and G_use.has_edge(int(r.u), int(r.v))]
        if not cand:
            rows.append(dict(city=ulmm["city"], vehicle=veh, orientation=ori,
                             T_med=np.nan, R_med=np.nan,
                             T_share=np.nan, R_share=np.nan,
                             T_med_aff=np.nan, R_med_aff=np.nan,
                             T_q75_aff=np.nan, R_q75_aff=np.nan))
            continue

        cand.sort(key=lambda x: x[2], reverse=True)
        n_rm = max(1, int(round(len(cand) * (budget_pct/100.0))))
        topE = set((u,v) for u,v,_ in cand[:n_rm])
        E_all = list(G_use.edges())

        tau0 = _nearest_access_times(G_use, a_nodes, ignore=None)
        T = _impact_stats(G_use, a_nodes, d_nodes, tau0, ignore=topE)

        R_list = [
            _impact_stats(G_use, a_nodes, d_nodes, tau0,
                          ignore=set(map(tuple, rng.choice(E_all, size=n_rm, replace=False))))
            for _ in range(reps)
        ]
        def agg(k): return float(np.nanmedian([r[k] for r in R_list])) if R_list else np.nan
        rows.append(dict(
            city=ulmm["city"], vehicle=veh, orientation=ori,
            T_med=T["med"],             R_med=agg("med"),
            T_share=T["share_aff"],     R_share=agg("share_aff"),
            T_med_aff=T["med_aff"],     R_med_aff=agg("med_aff"),
            T_q75_aff=T["q75_aff"],     R_q75_aff=agg("q75_aff"),
        ))
    return pd.DataFrame(rows)

# -----------------------------
# Plot rapidi (H2, H3)
# -----------------------------
def plot_h2(df_cov: pd.DataFrame, base_key: str, tau: int, out: Path):
    sub = df_cov[df_cov["tau"]==tau]
    if sub.empty: 
        return
    cities = sorted(sub["city"].unique())
    n = len(cities); ncols=2; nrows=int(np.ceil(n/ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(10, 2.6*nrows))
    axes = np.array(axes).reshape(nrows, ncols)
    for i, city in enumerate(cities):
        ax = axes[i//ncols, i%ncols]
        for ori in ORIENTATIONS:
            ss = sub[(sub.city==city)&(sub.orientation==ori)].sort_values("q")
            if ss.empty: 
                continue
            ax.plot(ss["q"], ss["cov_servc"], label=f"{ori} ServC")
            ax.plot(ss["q"], ss[base_key], linestyle="--", label=f"{ori} {base_key.split('_')[1]}")
        ax.set_title(f"{city} — τ={tau//60} min"); ax.set_xlabel("q"); ax.set_ylabel("Coverage")
        ax.set_ylim(0,1.02); ax.grid(True, linewidth=0.5, alpha=0.25)
    for j in range(i+1, nrows*ncols):
        axes[j//ncols, j%ncols].axis("off")
    h,l = axes[0,0].get_legend_handles_labels()
    if h: 
        fig.legend(h,l,loc="lower center", ncol=4)
    plt.tight_layout(rect=[0,0.04,1,0.97])
    fig.savefig(out, dpi=180); plt.close(fig)
    print("[H2] Saved", out)

def plot_h3(df, out, budget_pct=1.5):
    if df.empty: return
    agg = df.groupby("orientation").median(numeric_only=True).reindex(["AD","DA"])

    fig, axs = plt.subplots(1, 3, figsize=(13, 4))
    x = np.arange(len(agg))

    panels = [
        ("T_share","R_share","Affected demand (%)",0,100),
        ("T_med_aff","R_med_aff","Median Δτ | affected (%)",0,None),
        ("T_q75_aff","R_q75_aff","75th pct Δτ | affected (%)",0,None),
    ]
    for i,(kT,kR,ylabel,y0,y1fix) in enumerate(panels):
        axs[i].bar(x-0.15, agg[kT], width=0.3, label="Targeted")
        axs[i].bar(x+0.15, agg[kR], width=0.3, label="Random")
        # dots per-city
        for j,ori in enumerate(["AD","DA"]):
            yT = df.loc[df.orientation==ori, kT]; yR = df.loc[df.orientation==ori, kR]
            axs[i].scatter(np.full_like(yT, j-0.15), yT, s=12, alpha=0.35, zorder=3, edgecolor='none')
            axs[i].scatter(np.full_like(yR, j+0.15), yR, s=12, alpha=0.35, zorder=3, edgecolor='none')
        axs[i].set_xticks(x); axs[i].set_xticklabels(["AD","DA"])
        axs[i].set_ylabel(ylabel)
        ymax = y1fix if y1fix is not None else 1.1*float(np.nanmax([agg[kT].max(), agg[kR].max(), 1.0]))
        axs[i].set_ylim(y0, ymax)
        # valori sopra le barre
        for bar in axs[i].patches:
            h = bar.get_height()
            if np.isfinite(h) and h>0:
                axs[i].text(bar.get_x()+bar.get_width()/2, h+0.5, f"{h:.1f}", ha="center", va="bottom", fontsize=8)

    fig.suptitle(f"H3 (NAE-targeted), pooled across cities — budget p={budget_pct:g}%")
    fig.legend(loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.02))
    plt.tight_layout(rect=[0,0,1,0.92]); fig.savefig(out, dpi=180); plt.close(fig)

In [14]:
pickles = find_pickles()
if not pickles:
    print("No pickles in", PICKLES_DIR); 

all_nae = []
all_cov = []
all_shock = []

for p in pickles:
    ulmm = load_ulmm(p)
    if ulmm["access"].empty or ulmm["demand"].empty:
        print("[skip]", ulmm["city"], "(no access/demand)"); 
        continue
    print("[city]", ulmm["city"])

    # NAE per entrambe le orientazioni
    for ori in ORIENTATIONS:
        all_nae.append(nae_edge_scores(ulmm, VEHICLE, ori))

    # H2 coverage curves (ServC auto-calibrato)
    for ori in ORIENTATIONS:
        all_cov.append(
            coverage_curves(
                ulmm, VEHICLE, ori, taus=TAUS,
                qs=np.linspace(0.05,1.0,QS_POINTS),
                kappa_mode="auto", kappa_alpha=0.15,   # ≈15% del tempo mediano
                zeta=0.75, gamma_alpha=0.0
            )
        )

    # H3 shocks con NAE
    all_shock.append(shock_with_nae(ulmm, VEHICLE))

# ---- save & plot
if all_nae:
    df_nae = pd.concat(all_nae, ignore_index=True)
    df_nae.to_csv(RESULTS_DIR / f"nae_edges_{slugify(VEHICLE)}.csv", index=False)
    print("[save] NAE edges saved.")

if all_cov:
    df_cov = pd.concat(all_cov, ignore_index=True)
    df_cov.to_csv(RESULTS_DIR / f"h2_cov_{slugify(VEHICLE)}.csv", index=False)
    for tau in TAUS:
        plot_h2(df_cov, base_key="cov_b3",
                tau=tau, out=FIG_DIR / f"fig-servc-vsB3_tau{tau}_{slugify(VEHICLE)}.png")
        plot_h2(df_cov, base_key="cov_median",
                tau=tau, out=FIG_DIR / f"fig-servc-vsMedian_tau{tau}_{slugify(VEHICLE)}.png")

if all_shock:
    df_shock = pd.concat(all_shock, ignore_index=True)
    df_shock.to_csv(RESULTS_DIR / f"h3_shock_nae_{slugify(VEHICLE)}.csv", index=False)
    plot_h3(df_shock, FIG_DIR / f"fig-shock-NAE_{slugify(VEHICLE)}.png", budget_pct=BUDGET_PCT)

[city] Amsterdam, Netherlands


/var/folders/g0/xl_ttf4551zcc0vm60c7rgh40000gn/T/ipykernel_52289/534542247.py:371: RuntimeWarning: All-NaN slice encountered
  def agg(k): return float(np.nanmedian([r[k] for r in R_list])) if R_list else np.nan


[city] Barcelona, Spain


/var/folders/g0/xl_ttf4551zcc0vm60c7rgh40000gn/T/ipykernel_52289/534542247.py:371: RuntimeWarning: All-NaN slice encountered
  def agg(k): return float(np.nanmedian([r[k] for r in R_list])) if R_list else np.nan


[city] New York City, New York, USA


...

[city] Paris, France


/var/folders/g0/xl_ttf4551zcc0vm60c7rgh40000gn/T/ipykernel_52289/534542247.py:371: RuntimeWarning: All-NaN slice encountered
  def agg(k): return float(np.nanmedian([r[k] for r in R_list])) if R_list else np.nan


[city] Seattle, Washington, USA


/var/folders/g0/xl_ttf4551zcc0vm60c7rgh40000gn/T/ipykernel_52289/534542247.py:371: RuntimeWarning: All-NaN slice encountered
  def agg(k): return float(np.nanmedian([r[k] for r in R_list])) if R_list else np.nan


[save] NAE edges saved.
[H2] Saved fig/fig-servc-vsB3_tau300_van.png
[H2] Saved fig/fig-servc-vsMedian_tau300_van.png
[H2] Saved fig/fig-servc-vsB3_tau600_van.png
[H2] Saved fig/fig-servc-vsMedian_tau600_van.png
[H3] Saved fig/fig-shock-NAE_van.png


In [16]:
plot_h3(df_shock, FIG_DIR / f"fig-shock-NAE_{slugify(VEHICLE)}.png", budget_pct=BUDGET_PCT)